

### 1. General Steady Conservation Equations

For incompressible, two-dimensional flow, the continuous governing equations (Continuity, Momentum, and Energy) share a unified scalar transport form for a field property $\phi$:

$$\nabla \cdot (\rho \mathbf{u} \phi) = \nabla \cdot (\Gamma_\phi \nabla \phi) + S_\phi$$

```
                   North face (n)
              +----------------------+
              |                      |
              |       (N)            |
              |                      |
              |                      |
West face (w) | (W)   (P)   (E)      | East face (e)
              |                      |
              |       (S)            |
              |                      |
              +----------------------+
                   South face (s)

```

| Transport Variable ($\phi$) | Diffusion Coefficient ($\Gamma_\phi$) | Source Term ($S_\phi$) | Equation Name |
| --- | --- | --- | --- |
| $1$ | $0$ | $0$ | Mass Continuity |
| $u$ | $\mu$ | $-\frac{\partial p}{\partial x} + S_{m, x}$ | $x$-Momentum |
| $v$ | $\mu$ | $-\frac{\partial p}{\partial y} + S_{m, y}$ | $y$-Momentum |
| $T$ | $\frac{k}{c_p}$ | $\frac{q'''}{c_p}$ | Energy |

---

### 2. Control Volume Integration

Integrating the general transport equation over a discrete control volume $V_P = \Delta x \cdot \Delta y \cdot 1$ centered at node $P$:

$$\int_{V_P} \nabla \cdot (\rho \mathbf{u} \phi) \, dV = \int_{V_P} \nabla \cdot (\Gamma_\phi \nabla \phi) \, dV + \int_{V_P} S_\phi \, dV$$

Applying Gauss's Divergence Theorem converts volume integrals of gradients into surface flux integrals across control volume faces $e, w, n, s$:

$$(J_{e,\text{conv}} - J_{w,\text{conv}}) + (J_{n,\text{conv}} - J_{s,\text{conv}}) = (J_{e,\text{diff}} - J_{w,\text{diff}}) + (J_{n,\text{diff}} - J_{s,\text{diff}}) + S_{\phi, P} \Delta x \Delta y$$

---

### 3. Face Mass Fluxes & Diffusive Conductances

Define the convective mass flux $F$ and diffusive conductance $D$ across each control volume face:

**East Face ($e$):**


$$F_e = \rho u_e \Delta y, \qquad D_e = \frac{\Gamma_\phi \Delta y}{\Delta x}$$

**West Face ($w$):**


$$F_w = \rho u_w \Delta y, \qquad D_w = \frac{\Gamma_\phi \Delta y}{\Delta x}$$

**North Face ($n$):**


$$F_n = \rho v_n \Delta x, \qquad D_n = \frac{\Gamma_\phi \Delta x}{\Delta y}$$

**South Face ($s$):**


$$F_s = \rho v_s \Delta x, \qquad D_s = \frac{\Gamma_\phi \Delta x}{\Delta y}$$

---

### 4. Discretization Schemes

#### A. Diffusive Fluxes (Central Differencing Scheme - CDS)

Evaluated assuming linear variation of $\phi$ between cell centers:

$$J_{e,\text{diff}} = \left. \Gamma_\phi \frac{\partial \phi}{\partial x} \right\vert{}_e \Delta y \approx D_e (\phi_E - \phi_P)$$

$$J_{w,\text{diff}} = \left. \Gamma_\phi \frac{\partial \phi}{\partial x} \right\vert{}_w \Delta y \approx D_w (\phi_P - \phi_W)$$

#### B. Convective Fluxes (First-Order Upwind Differencing Scheme - UDS)

To ensure unconditional numerical stability and satisfy the transportiveness property in advection-dominated flows, face values are assigned based on the direction of fluid velocity:

$$\phi_e = \begin{cases} \phi_P & \text{if } F_e > 0 \text{ (Outflow from } P) \\ \phi_E & \text{if } F_e < 0 \text{ (Inflow to } P) \end{cases}$$

Using the identity $F_e \phi_e = \phi_P \max(F_e, 0) - \phi_E \max(-F_e, 0)$, the net east convective flux is:

$$J_{e,\text{conv}} = F_e \phi_e = \phi_P \max(F_e, 0) - \phi_E \max(-F_e, 0)$$

---

### 5. Linear Algebraic Equation Assembly

Substituting the discretized convective and diffusive flux terms into the integrated balance equation yields:

$$\Big[\phi_P \max(F_e, 0) - \phi_E \max(-F_e, 0)\Big] - \Big[\phi_W \max(F_w, 0) - \phi_P \max(-F_w, 0)\Big] + \Big[\phi_P \max(F_n, 0) - \phi_N \max(-F_n, 0)\Big] - \Big[\phi_S \max(F_s, 0) - \phi_P \max(-F_s, 0)\Big] = D_e(\phi_E - \phi_P) - D_w(\phi_P - \phi_W) + D_n(\phi_N - \phi_P) - D_s(\phi_P - \phi_S) + S_{\phi, P} \Delta x \Delta y$$

Rearranging terms in favor of cell-center property $\phi_P$ and neighbor nodes ($\phi_W, \phi_E, \phi_S, \phi_N$):

$$a_P \phi_P = a_W \phi_W + a_E \phi_E + a_S \phi_S + a_N \phi_N + b$$

Where neighbor influence coefficients are defined as:

$$a_W = D_w + \max(F_w, 0)$$

$$a_E = D_e + \max(-F_e, 0)$$

$$a_S = D_s + \max(F_s, 0)$$

$$a_N = D_n + \max(-F_n, 0)$$

$$b = S_{\phi, P} \Delta x \Delta y$$

---

### 6. Role of Mass Continuity in $a_P$ Coefficient Formulation

The discrete mass continuity equation over the same control volume enforces mass balance:

$$F_e - F_w + F_n - F_s = 0$$

Expanding the central coefficient $a_P$ gives:

$$a_P = a_W + a_E + a_S + a_N + (F_e - F_w + F_n - F_s)$$

When the velocity field satisfies continuity exactly, $(F_e - F_w + F_n - F_s) = 0$, simplifying the central coefficient to:

$$a_P = a_W + a_E + a_S + a_N$$

---

### 7. Direct Mapping to Thermal Energy Solver ($T$)

Setting $\phi = T$, $\Gamma_T = \frac{k}{c_p}$, and $S_T = \frac{q'''}{c_p}$ (or scaling mass fluxes directly by $c_p$), the discrete steady temperature equation solved at each cell $(i, j)$ is:

$$a_P T_{i,j} = a_W T_{i-1,j} + a_E T_{i+1,j} + a_S T_{i,j-1} + a_N T_{i,j+1} + q'''_{i,j} \Delta x \Delta y$$

This algebraic equation is updated iteratively across the 2D grid using the Gauss-Seidel method until the residual $\Vert{}T^{(k+1)} - T^{(k)}\Vert{}_\infty < 10^{-6}$.